## Imports

In [2]:
import pandas as pd
import numpy as np
import joblib
import dice_ml
from dice_ml import Dice

print("Bibliotheques importees.")

Bibliotheques importees.


## Paramètres

In [3]:
OUTPUT_DIR = "../../outputs/automobile"
TARGET_COL = "price"

# IMPORTANT : ces deux valeurs doivent correspondre exactement a celles
# utilisees dans 3_Contrefactuels.ipynb (Cellule 6), pour comparer les
# methodes sur la meme instance et la meme cible.
INDEX_INSTANCE = 21
DELTA          = 1485.77

SEUIL_VALIDITE = 0.01
NB_CF          = 7

## Chargement des fichiers (issus des notebooks 1 et 2)

In [4]:
print("[DiCE] Chargement du modele et des donnees...")

model = joblib.load(f"{OUTPUT_DIR}/svm_model.pkl")

X_train = pd.read_csv(f"{OUTPUT_DIR}/X_train.csv", sep=";")
y_train = pd.read_csv(f"{OUTPUT_DIR}/y_train.csv", sep=";")[TARGET_COL]
X_test  = pd.read_csv(f"{OUTPUT_DIR}/X_test.csv", sep=";")

# Dataset d'entrainement complet (features + cible), requis par dice_ml
# pour connaitre les plages de valeurs de chaque attribut.
train_data = X_train.copy()
train_data[TARGET_COL] = y_train.values

# L'instance a expliquer vient du test set (meme index que le notebook maison)
exemple = X_test.iloc[[INDEX_INSTANCE]]
fx = model.predict(exemple)[0]
Y_CIBLE = fx + DELTA

print(f"[DiCE] Prediction f(x) : {fx:.2f}")
print(f"[DiCE] Cible y'        : {Y_CIBLE:.2f}")

[DiCE] Chargement du modele et des donnees...
[DiCE] Prediction f(x) : 10514.23
[DiCE] Cible y'        : 12000.00


## Configuration de DiCE

In [5]:
print("[DiCE] Configuration de dice_ml...")

# DiCE a besoin : (1) des donnees d'ENTRAINEMENT (pour connaitre les plages
# de valeurs), (2) du modele, (3) du type de tache (ici "regression")
data_dice = dice_ml.Data(
    dataframe=train_data,
    continuous_features=[c for c in train_data.columns if c != TARGET_COL],
    outcome_name=TARGET_COL
)

model_dice = dice_ml.Model(
    model=model,
    backend="sklearn",
    model_type="regressor"
)

# Methode "genetic" : algorithme genetique, fonctionne avec n'importe quel
# modele boite noire
exp = Dice(data_dice, model_dice, method="genetic")
print("[DiCE] Pret a generer les contrefactuels.")

[DiCE] Configuration de dice_ml...


[DiCE] Pret a generer les contrefactuels.


## Génération des contrefactuels vers Y_CIBLE

In [6]:
print(f"[DiCE] Generation de {NB_CF} contrefactuels vers y' = {Y_CIBLE:.2f}...")

desired_range = [Y_CIBLE * 0.95, Y_CIBLE * 1.05]

dice_exp = exp.generate_counterfactuals(
    exemple,
    total_CFs=NB_CF,
    desired_range=desired_range,
    posthoc_sparsity_param=0,   # desactive la correction de parcimonie qui annulait tout
    verbose=False
)

dice_exp.visualize_as_dataframe(show_only_changes=True)

cf_check = dice_exp.cf_examples_list[0].final_cfs_df
print("\n[DiCE] Predictions des contrefactuels generes :")
print(cf_check[TARGET_COL].values)
print(f"[DiCE] Plage visee : {desired_range}")

[DiCE] Generation de 7 contrefactuels vers y' = 12000.00...


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:16<00:00, 16.61s/it]

Query instance (original outcome : 10514.232421875)


,normalized_losses,wheel_base,length,width,height,curb_weight,engine_size,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price
0,110.0,103.300003,174.600006,64.599998,59.799999,2535,122,3.34,3.46,8.5,88.0,5000.0,24,30,10514.232422



Diverse Counterfactual set (new outcome: [np.float64(11400.001567730693), np.float64(12600.001732754978)])


,normalized_losses,wheel_base,length,width,height,curb_weight,engine_size,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price
0,72.2,86.7,163.5,60.3,56.5,-,128.0,3.23,3.35,7.5,48.0,-,-,-,-
0,134.0,98.4,-,65.6,53.0,-,146.0,3.62,3.5,9.3,116.0,-,-,-,-
0,84.7,86.7,163.5,61.0,50.2,-,116.0,3.5,3.32,8.6,73.4,-,18.0,-,-
0,106.8,101.0,-,63.5,57.9,3608.0,159.0,3.22,3.52,7.9,81.6,-,39.0,42.0,-
0,84.7,86.6,163.5,61.0,50.2,1488.0,116.0,3.5,3.32,10.0,48.0,-,18.0,-,-
0,91.0,93.7,151.1,65.1,54.5,-,169.0,3.23,3.4,8.7,163.9,-,30.0,38.0,-
0,112.4,98.6,185.6,-,58.4,1488.0,61.0,3.54,3.36,7.5,48.0,-,26.0,47.0,-



[DiCE] Predictions des contrefactuels generes :
[11747.82561304 12138.63662698 12207.21250491 12381.82575981
 12364.72485204 12590.03899559 12537.86598825]
[DiCE] Plage visee : [np.float64(11400.001567730693), np.float64(12600.001732754978)]


## Extraction au format comparable à contrefactuels.csv

In [7]:
print("[DiCE] Mise en forme des resultats...")

cf_df = dice_exp.cf_examples_list[0].final_cfs_df.reset_index(drop=True)
x_orig = exemple.iloc[0]
rows = []

ligne_originale = {"motif": "-", "nb_sauts": "-", "support": "-", "pred_cible": "-"}
for col in exemple.columns:
    ligne_originale[col] = round(x_orig[col], 4)
ligne_originale[TARGET_COL] = round(fx, 2)
rows.append(ligne_originale)

for i, row in cf_df.iterrows():
    pred_cf = row[TARGET_COL]
    ligne = {"motif": "DiCE (genetic)", "nb_sauts": "-", "support": "-", "pred_cible": Y_CIBLE}
    for col in exemple.columns:
        val_cf = row[col]
        if abs(x_orig[col] - val_cf) > 1e-6:
            fleche = "↑" if val_cf > x_orig[col] else "↓"
            ligne[col] = f"{round(val_cf, 4)} ({fleche})"
        else:
            ligne[col] = "-"
    fleche_price = "↑" if pred_cf > fx else "↓"
    ligne[TARGET_COL] = f"{round(pred_cf, 2)} ({fleche_price})"
    rows.append(ligne)

df_cf_dice = pd.DataFrame(rows)
df_cf_dice.to_csv(f"{OUTPUT_DIR}/contrefactuels_dice.csv", index=False, sep=";")

print(f"[DiCE] Sauvegarde dans contrefactuels_dice.csv ({len(rows) - 1} contrefactuel(s) + 1 originale)")
print(df_cf_dice.to_string(index=False))

[DiCE] Mise en forme des resultats...
[DiCE] Sauvegarde dans contrefactuels_dice.csv (7 contrefactuel(s) + 1 originale)
         motif nb_sauts support   pred_cible normalized_losses wheel_base    length    width   height curb_weight engine_size     bore   stroke compression_ratio horsepower   peak_rpm city_mpg highway_mpg        price
             -        -       -            -             110.0      103.3     174.6     64.6     59.8      2535.0       122.0     3.34     3.46               8.5       88.0     5000.0     24.0        30.0     10514.23
DiCE (genetic)        -       -  12000.00165          72.2 (↓)   86.7 (↓) 163.5 (↓) 60.3 (↓) 56.5 (↓)  3255.0 (↑)   128.0 (↑) 3.23 (↓) 3.35 (↓)           7.5 (↓)   48.0 (↓) 5782.8 (↑)        -           - 11747.83 (↑)
DiCE (genetic)        -       -  12000.00165         134.0 (↑)   98.4 (↓) 176.2 (↑) 65.6 (↑) 53.0 (↓)  2975.0 (↑)   146.0 (↑) 3.62 (↑)  3.5 (↑)           9.3 (↑)  116.0 (↑) 4800.0 (↓)        -           - 12138.64 (↑)
DiCE (ge

## Métriques

In [8]:
def calculer_metriques(x_orig, contrefactuels, fx_cible, seuil_validite=0.01):
    """Memes formules ET meme calcul du MAD que calculer_metriques() du
    notebook 3_Contrefactuels.ipynb (MAD calcule sur x_orig, l'instance elle-meme)."""
    x0 = x_orig.iloc[0]
    n = len(contrefactuels)
    if n == 0:
        return None

    mad = x_orig.apply(lambda col: np.median(np.abs(col - col.median())), axis=0)
    mad = mad.replace(0, 1)

    nb_valides = sum(1 for cf in contrefactuels if abs(cf["pred_cf"] - fx_cible) < seuil_validite * abs(fx_cible))
    validite = nb_valides / n

    proximites = [np.mean([abs(x0[c] - cf["contrefactuel"][c]) / mad[c] for c in x0.index]) for cf in contrefactuels]
    proximite = np.mean(proximites)

    nb_cols = len(x0)
    sparsites = [1 - sum(1 for c in x0.index if abs(x0[c] - cf["contrefactuel"][c]) > 1e-6) / nb_cols
                 for cf in contrefactuels]
    sparsite = np.mean(sparsites)

    if n < 2:
        diversite = 0.0
    else:
        paires = []
        for i in range(n):
            for j in range(i + 1, n):
                xi, xj = contrefactuels[i]["contrefactuel"], contrefactuels[j]["contrefactuel"]
                paires.append(np.mean([abs(xi[c] - xj[c]) / mad[c] for c in x0.index]))
        diversite = np.mean(paires)

    return {"nb": n, "validite": round(float(validite), 4), "proximite": round(float(proximite), 4),
            "sparsite": round(float(sparsite), 4), "diversite": round(float(diversite), 4)}


contrefactuels = [
    {"pred_cf": cf_df.iloc[i][TARGET_COL], "contrefactuel": cf_df.iloc[i].drop(TARGET_COL)}
    for i in range(len(cf_df))
]

m = calculer_metriques(exemple, contrefactuels, Y_CIBLE, SEUIL_VALIDITE)

print("=" * 50)
print("  METRIQUES DU CONTREFACTUEL (DiCE)")
print("-" * 50)
for k, v in m.items():
    print(f"  {k:<12}: {v}")

  METRIQUES DU CONTREFACTUEL (DiCE)
--------------------------------------------------
  nb          : 7
  validite    : 0.0
  proximite   : 104.1121
  sparsite    : 0.0714
  diversite   : 132.2027
